In [2]:
import pandas as pd

In [3]:
# Load CATH domain classification data
df = pd.read_csv('../data/cath-classification-data/cath-domain-list.txt', 
                 sep=r'\s+', comment='#', engine='python',
                 names=['domain_name', 'class_number', 'architecture_number', 'topology_number',
                        'homologous_superfamily_number', 's35_sequence_cluster_number',
                        's60_sequence_cluster_number', 's95_sequence_cluster_number',
                        's100_sequence_cluster_number', 's100_sequence_count_number',
                        'domain_length', 'structure_resolution'])

# Create CATH code (class.architecture.topology.homologous_superfamily)
df['cath_code'] = (df['class_number'].astype(str) + '.' + 
                   df['architecture_number'].astype(str) + '.' + 
                   df['topology_number'].astype(str) + '.' + 
                   df['homologous_superfamily_number'].astype(str))

# Save domain to CATH code mapping
df[['domain_name', 'cath_code']].to_csv('../data/cath-domain-sf-list.txt', 
                                        sep='\t', header=False, index=False)

print(f"Loaded {len(df)} CATH domain entries")
df.head()


Loaded 601328 CATH domain entries


,domain_name,class_number,architecture_number,topology_number,homologous_superfamily_number,s35_sequence_cluster_number,s60_sequence_cluster_number,s95_sequence_cluster_number,s100_sequence_cluster_number,s100_sequence_count_number,domain_length,structure_resolution,cath_code
0,1oaiA00,1,10,8,10,1,1,1,1,1,59,1.0,1.10.8.10
1,1go5A00,1,10,8,10,1,1,1,1,2,69,999.0,1.10.8.10
2,3frhA01,1,10,8,10,2,1,1,1,1,58,1.2,1.10.8.10
3,3friA01,1,10,8,10,2,1,1,1,2,54,1.8,1.10.8.10
4,3b89A01,1,10,8,10,2,1,1,2,1,54,2.6,1.10.8.10


In [ ]:
# Load FASTA sequences into a dataframe
from Bio import SeqIO
import io

def load_fasta_to_dataframe(fasta_file):
    """Load FASTA file into a pandas DataFrame with sequence headers and sequences."""
    sequences = []
    headers = []
    
    with open(fasta_file, 'r') as handle:
        for record in SeqIO.parse(handle, "fasta"):
            headers.append(record.id)
            sequences.append(str(record.seq))
    
    return pd.DataFrame({
        'header': headers,
        'sequence': sequences
    })

# Load the CATH domain sequences - main sequence datasets
seq_df_100 = load_fasta_to_dataframe('../data/sequence-data/cath-domain-seqs-S100.fa')
seq_df_95 = load_fasta_to_dataframe('../data/sequence-data/cath-domain-seqs-S95.fa')
seq_df_60 = load_fasta_to_dataframe('../data/sequence-data/cath-domain-seqs-S60.fa')
seq_df_35 = load_fasta_to_dataframe('../data/sequence-data/cath-domain-seqs-S35.fa')

# Load the non-redundant datasets separately
seq_df_40 = load_fasta_to_dataframe('../data/non-redundant-data-sets/cath-dataset-nonredundant-S40.fa')
seq_df_20 = load_fasta_to_dataframe('../data/non-redundant-data-sets/cath-dataset-nonredundant-S20.fa')

print("Main sequence datasets:")
print(f"Loaded {len(seq_df_100)} sequences from s100 FASTA file")
print(f"Loaded {len(seq_df_95)} sequences from s95 FASTA file")
print(f"Loaded {len(seq_df_60)} sequences from s60 FASTA file")
print(f"Loaded {len(seq_df_35)} sequences from s35 FASTA file")

print("\nNon-redundant datasets:")
print(f"Loaded {len(seq_df_40)} sequences from s40 FASTA file")
print(f"Loaded {len(seq_df_20)} sequences from s20 FASTA file")

def extract_domain_id(header):
    """Extract domain ID from FASTA header (e.g., 'cath|4_4_0|101mA00/0-153' -> '101mA00')"""
    if '|' in header:
        return header.split('|')[2].split('/')[0]
    return header.split('/')[0]

# Extract domain IDs from main sequence datasets
domains_100 = set(seq_df_100['header'].apply(extract_domain_id))
domains_95 = set(seq_df_95['header'].apply(extract_domain_id))
domains_60 = set(seq_df_60['header'].apply(extract_domain_id))
domains_35 = set(seq_df_35['header'].apply(extract_domain_id))

# Extract domain IDs from non-redundant datasets
domains_40 = set(seq_df_40['header'].apply(extract_domain_id))
domains_20 = set(seq_df_20['header'].apply(extract_domain_id))

print("\nChecking subset relationships for main sequence datasets:")
print(f"S95 ⊆ S100: {domains_95.issubset(domains_100)} ({len(domains_95)} ⊆ {len(domains_100)})")
print(f"S60 ⊆ S95:  {domains_60.issubset(domains_95)} ({len(domains_60)} ⊆ {len(domains_95)})")
print(f"S35 ⊆ S60:  {domains_35.issubset(domains_60)} ({len(domains_35)} ⊆ {len(domains_60)})")

print("\nChecking subset relationships for non-redundant datasets:")
print(f"S20 ⊆ S40:  {domains_20.issubset(domains_40)} ({len(domains_20)} ⊆ {len(domains_40)})")
print(f"S20 ⊆ S100: {domains_20.issubset(domains_100)} ({len(domains_20)} ⊆ {len(domains_100)})")
print(f"S40 ⊆ S100: {domains_40.issubset(domains_100)} ({len(domains_40)} ⊆ {len(domains_100)})")

print("\nChecking relationships between main and non-redundant datasets:")
print(f"S40 ⊆ S60:  {domains_40.issubset(domains_60)} ({len(domains_40)} ⊆ {len(domains_60)})")
print(f"S20 ⊆ S35:  {domains_20.issubset(domains_35)} ({len(domains_20)} ⊆ {len(domains_35)})")

# Show any domains that break the subset relationship for main datasets
if not domains_95.issubset(domains_100):
    diff = domains_95 - domains_100
    print(f"\nDomains in S95 but not S100: {len(diff)} domains")
    print(f"Examples: {list(diff)[:5]}")
if not domains_60.issubset(domains_95):
    diff = domains_60 - domains_95
    print(f"Domains in S60 but not S95: {len(diff)} domains")
    print(f"Examples: {list(diff)[:5]}")
if not domains_35.issubset(domains_60):
    diff = domains_35 - domains_60
    print(f"Domains in S35 but not S60: {len(diff)} domains")
    print(f"Examples: {list(diff)[:5]}")

# Show any domains that break the subset relationship for non-redundant datasets
if not domains_20.issubset(domains_40):
    diff = domains_20 - domains_40
    print(f"\nDomains in S20 but not S40: {len(diff)} domains")
    print(f"Examples: {list(diff)[:5]}")
if not domains_40.issubset(domains_100):
    diff = domains_40 - domains_100
    print(f"Domains in S40 but not S100: {len(diff)} domains")
    print(f"Examples: {list(diff)[:5]}")
if not domains_20.issubset(domains_100):
    diff = domains_20 - domains_100
    print(f"Domains in S20 but not S100: {len(diff)} domains")
    print(f"Examples: {list(diff)[:5]}")

# Show any domains that break the cross-dataset relationships
if not domains_40.issubset(domains_100):
    diff = domains_40 - domains_100
    print(f"\nDomains in S40 but not S100: {len(diff)} domains")
    print(f"Examples: {list(diff)[:5]}")
if not domains_20.issubset(domains_100):
    diff = domains_20 - domains_100
    print(f"Domains in S20 but not S100: {len(diff)} domains")
    print(f"Examples: {list(diff)[:5]}")


Main sequence datasets:
Loaded 123251 sequences from s100 FASTA file
Loaded 73535 sequences from s95 FASTA file
Loaded 52974 sequences from s60 FASTA file
Loaded 37350 sequences from s35 FASTA file

Non-redundant datasets:
Loaded 34653 sequences from s40 FASTA file
Loaded 15043 sequences from s20 FASTA file

Checking subset relationships for main sequence datasets:
S95 ⊆ S100: True (73535 ⊆ 123251)
S60 ⊆ S95:  True (52974 ⊆ 73535)
S35 ⊆ S60:  True (37350 ⊆ 52974)

Checking subset relationships for non-redundant datasets:
S20 ⊆ S40:  False (15043 ⊆ 34653)
S20 ⊆ S100: True (15043 ⊆ 123251)
S40 ⊆ S100: True (34653 ⊆ 123251)

Checking relationships between main and non-redundant datasets:
S40 ⊆ S60:  False (34653 ⊆ 52974)
S40 ⊆ S35:  False (34653 ⊆ 37350)
S20 ⊆ S35:  False (15043 ⊆ 37350)

Domains in S20 but not S40: 4486 domains
Examples: ['5usfB01', '4aybB05', '6n9lA01', '3ipfA01', '4ciuA03']
